# VIKTOR App to Pydantic Workbench

This notebook walks through the full pipeline:

1. Input one or more VIKTOR app URLs.
2. Load `.env` and the repo helpers.
3. Fetch entity, entity type, editor session, and parametrization.
4. Validate the raw API responses with Pydantic.
5. Normalize the parametrization tree and discovered methods.
6. Build default payloads from `/parametrization/`.
7. Send those defaults back to the API as a validation step.
8. Probe discovered methods through `/jobs/`.
9. Optionally run the multi-app bench.

Note: this notebook depends on live VIKTOR demo endpoints. If `/parametrization/` is timing out, cells that hit the API will fail with a remote `504`.

In [ ]:
from pathlib import Path
import json
import os
import sys
from pprint import pprint

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / ".env").exists() and (PROJECT_ROOT.parent / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

WORKBENCH_DIR = PROJECT_ROOT / "vk-params-pydantic"
SRC_DIR = WORKBENCH_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_ROOT / ".env")

print({
    "project_root": str(PROJECT_ROOT),
    "workbench_dir": str(WORKBENCH_DIR),
    "src_dir": str(SRC_DIR),
    "token_configured": bool((os.getenv("TOKEN_VK_APP") or "").strip()),
    "api_base": os.getenv("VIKTOR_API_BASE", "https://demo.viktor.ai/api"),
})

In [ ]:
from vk_params_pydantic.bench import (
    ViktorParametrizationClient,
    build_app_config,
    build_config_for_target,
    build_default_payload_candidate,
    build_target_from_app_url,
    build_summary,
    extract_methods,
    normalize_iteration,
    normalize_param_node,
    probe_methods,
    run_parametrization_bench,
    validate_payload_candidate,
)
from vk_params_pydantic.models import ParametrizationScenario

In [ ]:
APP_URLS = [
    "https://demo.viktor.ai/workspaces/2232/app/editor/11640",
    "https://demo.viktor.ai/workspaces/2141/app/editor/11536",
]

targets = [build_target_from_app_url(url) for url in APP_URLS]
pprint([target.model_dump() for target in targets])

## Single App Walkthrough

Pick one target and inspect the raw API payloads first.

In [ ]:
selected_target = targets[0]
config = build_config_for_target(selected_target)
client = ViktorParametrizationClient(config)

entity = client.get_entity()
entity_type = client.get_entity_type(entity.entity_type)
editor_session = client.create_editor_session()
parametrization = client.get_parametrization(
    str(editor_session.editor_session),
    params=entity.properties,
)

{
    "target": selected_target.model_dump(),
    "entity_name": entity.name,
    "entity_type": entity_type.name,
    "saved_param_root_keys": sorted((entity.properties or {}).keys()),
    "view_methods": [view.controller_method for view in entity_type.views if view.controller_method],
}

In [ ]:
print("Entity response type:", type(entity).__name__)
print("Entity type response type:", type(entity_type).__name__)
print("Parametrization response type:", type(parametrization).__name__)
print()
print("Entity snippet:")
print(json.dumps(entity.model_dump(mode="json"), indent=2)[:2000])
print()
print("Parametrization snippet:")
print(json.dumps(parametrization.model_dump(mode="json"), indent=2)[:3000])

## Normalize the Parametrization Tree

This converts the raw resolved parametrization into generic Pydantic-backed container/field metadata.

In [ ]:
parametrization_tree = [
    normalize_param_node(node, entity.param_types)
    for node in parametrization.content.parametrization
]
methods = extract_methods(entity_type, parametrization)
summary = build_summary(parametrization_tree, methods)

{
    "summary": summary.model_dump(),
    "methods": [method.model_dump() for method in methods],
    "first_node": parametrization_tree[0].model_dump(mode="json") if parametrization_tree else None,
}

## Build Default Payloads From `/parametrization/`

Here we ask VIKTOR for the resolved defaults using `params={}` and then build:

- `defaults_only`
- `defaults_plus_saved`


In [ ]:
defaults_session = client.create_editor_session()
defaults_parametrization = client.get_parametrization(
    str(defaults_session.editor_session),
    params={},
)

default_payload = build_default_payload_candidate(
    defaults_parametrization,
    entity.properties,
    name="defaults_only",
    backfill_missing=False,
)
default_backfilled_payload = build_default_payload_candidate(
    defaults_parametrization,
    entity.properties,
    name="defaults_plus_saved",
    backfill_missing=True,
)

{
    "defaults_only": default_payload.model_dump(mode="json"),
    "defaults_plus_saved": default_backfilled_payload.model_dump(mode="json"),
}

## Validate the Generated Defaults Back Through the API

This sends the generated payloads back to `/parametrization/` to verify they are accepted.

In [ ]:
baseline_iteration = normalize_iteration(
    entity=entity,
    entity_type=entity_type,
    editor_session=str(editor_session.editor_session),
    scenario=ParametrizationScenario(
        name="baseline",
        description="Saved entity properties",
        overrides={},
    ),
    params_used=entity.properties,
    parametrization=parametrization,
)

default_validation = validate_payload_candidate(
    client=client,
    entity=entity,
    entity_type=entity_type,
    baseline_iteration=baseline_iteration,
    candidate=default_payload,
)
default_backfilled_validation = validate_payload_candidate(
    client=client,
    entity=entity,
    entity_type=entity_type,
    baseline_iteration=baseline_iteration,
    candidate=default_backfilled_payload,
)

{
    "defaults_only_validation": default_validation.model_dump(mode="json"),
    "defaults_plus_saved_validation": default_backfilled_validation.model_dump(mode="json"),
}

## Probe Discovered Methods Through `/jobs/`

This uses the validated `defaults_plus_saved` payload to test the discovered methods.

In [ ]:
method_probes = probe_methods(
    client=client,
    methods=methods,
    params=default_backfilled_payload.params or entity.properties,
)

[probe.model_dump(mode="json") for probe in method_probes]

## Run the Bench Across All Input App URLs

This reuses the repo bench helper and writes artifacts under `vk-params-pydantic/artifacts/notebook_bench/`.

In [ ]:
bench_report = run_parametrization_bench(
    targets=targets,
    max_auto_scenarios=1,
    output_root=PROJECT_ROOT / "vk-params-pydantic" / "artifacts" / "notebook_bench",
)

{
    "apps": [
        {
            "target": app.target.model_dump(),
            "baseline": app.loop_report.baseline.summary.model_dump(),
            "default_validation_success": app.loop_report.default_validation.success if app.loop_report.default_validation else None,
            "method_probes": [probe.model_dump(mode="json") for probe in app.loop_report.method_probes],
            "output_dir": app.output_dir,
        }
        for app in bench_report.apps
    ]
}